# 05 — Centrality and bridge-biomarker analysis

Node-centrality (strength, degree, betweenness) and bridge-centrality (inter-domain strength/degree) measures on the primary network (Methods 2.9, 2.10) — the primary endpoint of the study.

## Environment setup

In [ ]:
import os, sys, subprocess
from pathlib import Path


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "scripts").exists() and (PROJECT_ROOT / "outputs").exists()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if _in_colab():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "scikit-learn", "networkx", "statsmodels", "openpyxl"], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts._pipeline_common import *  # noqa: F401,F403

print("Project root:", PROJECT_ROOT)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fd = pd.read_csv(PROJECT_ROOT / 'outputs' / '01_ingest_harmonize' / 'feature_dictionary_v2.csv')
primary = fd.loc[fd.included_primary, 'feature'].tolist()
cols = primary

edges = pd.read_csv(PROJECT_ROOT / 'outputs' / '04_primary_graphical_lasso' / 'primary_graphical_lasso_partial_edges.csv')
metrics, G = graph_metrics(edges, {f: DOM[f] for f in cols})

out_dir = PROJECT_ROOT / 'outputs' / '05_centrality_bridge'
out_dir.mkdir(parents=True, exist_ok=True)
metrics.to_csv(out_dir / 'Table_primary_centrality_bridge_metrics.csv', index=False)
metrics.head(10).to_csv(out_dir / 'top_bridge_biomarkers_primary.csv', index=False)
metrics.sort_values('strength', ascending=False).head(10).to_csv(out_dir / 'top_central_biomarkers_primary.csv', index=False)

domain_edges = edges.groupby(['source_domain', 'target_domain']).size().reset_index(name='n_edges')
domain_edges.to_csv(out_dir / 'domain_connectivity_summary_primary.csv', index=False)

print('Top 8 bridge biomarkers:')
metrics.head(8)


### Figures

In [ ]:
for col, fn, title in [
    ('bridge_strength', 'Figure_top_bridge_strength_primary.png', 'Top bridge biomarkers'),
    ('strength', 'Figure_top_node_strength_primary.png', 'Top central biomarkers'),
]:
    top = metrics.sort_values(col, ascending=False).head(12).iloc[::-1]
    plt.figure(figsize=(8, 5)); plt.barh(top.feature, top[col]); plt.title(title); plt.tight_layout()
    plt.savefig(out_dir / fn, dpi=200); plt.show()
